In [1]:
import pandas as pd
import numpy as np
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier

c:\Users\VP678WV\OneDrive - EY\Documents\Delivery_Delay\logistic_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load train and test
train_df = pd.read_csv("C:\\Users\\VP678WV\\OneDrive - EY\\Documents\\Delivery_Delay\\data\\processed\\train_encoded2.csv")
test_df = pd.read_csv("C:\\Users\\VP678WV\\OneDrive - EY\\Documents\\Delivery_Delay\\data\\processed\\test_encoded2.csv")

In [3]:
# ----------------------
# 1. Prepare Data
# ----------------------
# Replace 'target' with actual target column
X_train = train_df.drop('target', axis=1)
y_train = train_df['target']

X_test = test_df.drop('target', axis=1)
y_test = test_df['target']

y_train = y_train.astype('category')
y_test = y_test.astype('category')

In [8]:
label_mapping = {-1: 0, 0: 1, 1: 2}

y_train = y_train.map(label_mapping)
y_test = y_test.map(label_mapping)

In [9]:
# ----------------------
# 2. Feature Selection using Random Forest
# ----------------------
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

feature_importances = pd.Series(rf.feature_importances_, index=X_train.columns)
top_features = feature_importances.sort_values(ascending=False)

# Select top N features (e.g., top 21)
selected_features = top_features.head(21).index.tolist()
selected_features

['order_performance_score',
 'order_to_shipment_planned_days',
 'shipping_mode',
 'customer_performance_score',
 'distance_normalized',
 'order_item_discount',
 'order_profit_per_order',
 'profit_per_order',
 'order_item_profit_ratio',
 'sales',
 'shipment_delay_days',
 'order_item_product_price',
 'shipping_hour_cos',
 'order_hour_cos',
 'shipping_hour_sin',
 'order_hour_sin',
 'shipping_dayofweek_sin',
 'order_shipping_time',
 'order_dayofweek_sin',
 'order_item_quantity',
 'order_to_shipment_days']

In [10]:
X_train_reduced = X_train[selected_features]
X_test_reduced = X_test[selected_features]

In [11]:
# ----------------------
# 3. Optuna Objective for XGBoost
# ----------------------
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
        'random_state': 42,
        'n_jobs': -1,
        'use_label_encoder': False,
        'eval_metric': 'mlogloss'  # Avoids warning
    }
    
    model = XGBClassifier(**params)
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    score = cross_val_score(
        model,
        X_train_reduced,
        y_train,
        cv=cv,
        scoring='f1_macro',
        n_jobs=-1
    )
    
    return score.mean()

In [12]:
# ----------------------
# 4. Run Optimization
# ----------------------
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, show_progress_bar=True)

print("Best Score:", study.best_value)
print("Best Params:", study.best_params)

[I 2025-08-12 13:19:06,156] A new study created in memory with name: no-name-d7347760-a0fb-4522-bb3f-0869eb1b01a1
Best trial: 0. Best value: 0.696419:   1%|          | 1/100 [00:15<25:54, 15.70s/it]

[I 2025-08-12 13:19:21,856] Trial 0 finished with value: 0.6964189475909773 and parameters: {'n_estimators': 694, 'max_depth': 4, 'learning_rate': 0.04270093420082193, 'subsample': 0.7373579179780843, 'colsample_bytree': 0.5534699315948297, 'gamma': 2.4564471430287464, 'reg_alpha': 0.6146994271052542, 'reg_lambda': 0.25137368628209755}. Best is trial 0 with value: 0.6964189475909773.


Best trial: 0. Best value: 0.696419:   2%|▏         | 2/100 [00:20<15:27,  9.47s/it]

[I 2025-08-12 13:19:26,963] Trial 1 finished with value: 0.684135815364476 and parameters: {'n_estimators': 519, 'max_depth': 10, 'learning_rate': 0.0892387433021595, 'subsample': 0.5759777995149679, 'colsample_bytree': 0.9873727651182571, 'gamma': 4.726694595037415, 'reg_alpha': 1.1338519375972829, 'reg_lambda': 3.338397424250721}. Best is trial 0 with value: 0.6964189475909773.


Best trial: 0. Best value: 0.696419:   3%|▎         | 3/100 [00:22<09:35,  5.93s/it]

...

[I 2025-08-12 14:41:45,392] Trial 97 finished with value: 0.7496441389167021 and parameters: {'n_estimators': 982, 'max_depth': 12, 'learning_rate': 0.029704640238411868, 'subsample': 0.5440275250510009, 'colsample_bytree': 0.966201209931206, 'gamma': 0.35346896136761496, 'reg_alpha': 0.6122400271412832, 'reg_lambda': 0.8949501483924881}. Best is trial 43 with value: 0.7571951037402507.


Best trial: 43. Best value: 0.757195:  99%|█████████▉| 99/100 [1:22:56<00:27, 27.29s/it]

[I 2025-08-12 14:42:02,877] Trial 98 finished with value: 0.7413652119329065 and parameters: {'n_estimators': 952, 'max_depth': 13, 'learning_rate': 0.02524758349795124, 'subsample': 0.5839936833006295, 'colsample_bytree': 0.6475351822002688, 'gamma': 0.604561724909999, 'reg_alpha': 0.4875514429668638, 'reg_lambda': 4.100754218105816}. Best is trial 43 with value: 0.7571951037402507.


Best trial: 43. Best value: 0.757195: 100%|██████████| 100/100 [1:23:26<00:00, 50.06s/it]

[I 2025-08-12 14:42:32,228] Trial 99 finished with value: 0.7541680622062032 and parameters: {'n_estimators': 733, 'max_depth': 12, 'learning_rate': 0.018575730472812683, 'subsample': 0.5538811866512046, 'colsample_bytree': 0.9858111402748909, 'gamma': 0.2093431842561627, 'reg_alpha': 0.2340001093738231, 'reg_lambda': 0.22872311894638225}. Best is trial 43 with value: 0.7571951037402507.
Best Score: 0.7571951037402507
Best Params: {'n_estimators': 859, 'max_depth': 14, 'learning_rate': 0.017578265956285696, 'subsample': 0.7445205736294263, 'colsample_bytree': 0.9985457633151822, 'gamma': 0.03961247433820425, 'reg_alpha': 0.04745791469362093, 'reg_lambda': 1.7290794318081057}


In [13]:
# ----------------------
# 5. Train Final Model
# ----------------------
best_params = study.best_params
best_params.update({
    'random_state': 42,
    'n_jobs': -1,
    'use_label_encoder': False,
    'eval_metric': 'mlogloss'
})

xgb_best = XGBClassifier(**best_params)
xgb_best.fit(X_train_reduced, y_train)

c:\Users\VP678WV\OneDrive - EY\Documents\Delivery_Delay\logistic_venv\lib\site-packages\xgboost\training.py:183: UserWarning: [14:43:32] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9985457633151822, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='mlogloss', feature_types=None, feature_weights=None,
              gamma=0.03961247433820425, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.017578265956285696,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=14, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=859, n_jobs=-1,
              num_parallel_tree=None, ...)

In [14]:
# ----------------------
# 6. Test Evaluation
# ----------------------
y_pred = xgb_best.predict(X_test_reduced)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test F1 Macro:", f1_score(y_test, y_pred, average='macro'))

Test Accuracy: 0.6167202572347267
Test F1 Macro: 0.5642681149838092


In [15]:
best_params

{'n_estimators': 859,
 'max_depth': 14,
 'learning_rate': 0.017578265956285696,
 'subsample': 0.7445205736294263,
 'colsample_bytree': 0.9985457633151822,
 'gamma': 0.03961247433820425,
 'reg_alpha': 0.04745791469362093,
 'reg_lambda': 1.7290794318081057,
 'random_state': 42,
 'n_jobs': -1,
 'use_label_encoder': False,
 'eval_metric': 'mlogloss'}